Fine-grained phonetic content decoding on electrodes we know are interesting (pre-filtered from behavior decoding analysis).

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from pathlib import Path
import torch
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
tqdm.pandas()
import mne

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.data import add_metadata_features
from src.models.decoding import run_decoding_searchlight_single_electrode, get_ensemble_predictions

In [ ]:
sns.set_context("paper", font_scale=2)

In [ ]:
subject = "EC279"
summary_path = {
    k: Path(f"outputs/causal4/behavior_decoding_single_electrode_summarize/{subject}/{k}_final_summary.csv")
    for k in ["A_early", "A"]
}

epochs_path = Path(f"outputs/epochs_preprocessed/{subject}_epo.fif")
electrodes_path = Path(f"outputs/causal4/find_speech_responsive/{subject}_results.csv")
outdir = "."

In [ ]:
subject = Path(electrodes_path).name.split("_")[0]

In [ ]:
sum_dfs = {
    k: pd.read_csv(path).set_index(["subject", "electrode_idx"])
    for k, path in summary_path.items()
}

In [ ]:
epochs = mne.read_epochs(epochs_path, preload=True, verbose=False)
epochs.metadata = add_metadata_features(epochs.metadata)

In [ ]:
electrode_df = pd.read_csv(electrodes_path).set_index("electrode_idx")

In [ ]:
# join early and late A behavioral results
A_df = pd.concat([sum_dfs["A_early"], sum_dfs["A"]])

In [ ]:
# Run a full searchlight on just the As we care about
study_As = A_df.reset_index() # A_df.query("diff > 1e-2").sort_values("stimulus_correlation", ascending=False).reset_index()

In [ ]:
searchlight_electrode_df = pd.merge(electrode_df, study_As.reset_index()[["electrode_idx"]].drop_duplicates(),
                                    left_index=True, right_on=["electrode_idx"], how="inner")
searchlight_electrode_df

In [ ]:
stimulus_decoding_min_sample = epochs.time_as_index(0.0)[0]
stimulus_decoding_max_sample = epochs.times.shape[0]
stimulus_decoding_window_size = 15
stimulus_decoding_stride = 5

In [ ]:
train_scores, test_scores, outcomes, models = run_decoding_searchlight_single_electrode(
    epochs={subject: epochs[epochs.metadata[epochs.metadata.resampled.isin((1, 6))].index]},
    global_min_sample=stimulus_decoding_min_sample,
    global_max_sample=stimulus_decoding_max_sample,
    electrode_df=searchlight_electrode_df,
    target="acoustic",
    smoke_test=False,
    strategy="train-test",
    window_size=stimulus_decoding_window_size,
    stride=stimulus_decoding_stride,
)

In [ ]:
scores_df = pd.concat(
    {key: pd.DataFrame(scores_i) for key, scores_i in test_scores.items()},
    names=["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])
scores_df

In [ ]:
# average over folds/repeats
avg_scores_df = scores_df.groupby(["subject", "electrode_idx", "phoneme_pair", "smin", "smax"]).mean().sort_values("roc_auc", ascending=False)
avg_scores_df

In [ ]:
# merge back in behav decoding information
merged_df = pd.merge(
    avg_scores_df.reset_index(),
    study_As[["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "word_end", "baseline_roc_auc", "full_roc_auc", "diff"]]
        .rename(columns={"smax": "behavior_smax", "smin": "behavior_smin", "baseline_roc_auc": "behavior_baseline_roc_auc",
                         "full_roc_auc": "behavior_full_roc_auc", "diff": "behavior_diff"}),
    on=["subject", "electrode_idx", "phoneme_pair"], how="inner"
)

## Evaluate on all epochs

In [ ]:
all_outcomes = {}

for key, models_i in tqdm(models.items()):
    # get predictions on all epochs
    epreds = get_ensemble_predictions(key, models_i, {subject: epochs})

    # Sanity check: for epochs covered in both analyses, we should see the same prediction values
    sanity_check_df = pd.merge(epreds, outcomes[key].sort_values("epoch_idx"), on=["epoch_idx", "fold"], how="inner")
    pd.testing.assert_series_equal(sanity_check_df.decoder_proba_x, sanity_check_df.decoder_proba_y, check_names=False)

    all_outcomes[key] = epreds

In [ ]:
torch.save({
    "train_scores": train_scores,
    "test_scores": test_scores,

    # model predictions on test folds
    "outcomes": outcomes,

    # model predictions on all relevant epochs for a given decoder (e.g. all p/b epochs for a p/b decoder)
    "all_outcomes": all_outcomes,

    "models": models,
    "scores_df": scores_df,
    "avg_scores_df": avg_scores_df,
}, Path(outdir) / "results.pt")

### Rolling window transfer analysis

In [ ]:
# rolling_transfer_results = []
# rolling_stride = 1
# for (subject, electrode_idx, phoneme_pair), rows in tqdm(A_df.groupby(["subject", "electrode_idx", "phoneme_pair"])):
#     ep_i = epochs[subject]
#     md_i = ep_i.metadata
#     data_i = ep_i.get_data()  # shape (n_epochs, n_channels, n_times)

#     study_smin, study_smax = 40, 70
#     study_key = (subject, electrode_idx, phoneme_pair, study_smin, study_smax)

#     study_dec_outcomes = A_individual_stimulus_decoders[subject]["outcomes"][study_key]

#     # First reproduce the original results
#     study_pipes = A_individual_stimulus_decoders[subject]["models"][study_key]
#     assert len(study_pipes) == study_dec_outcomes.fold.nunique()

#     for study_fold in range(len(study_pipes)):
#         study_previous_outcomes = study_dec_outcomes[study_dec_outcomes.fold == study_fold]
#         study_idxs = study_previous_outcomes.epoch_idx
#         study_data = data_i[study_idxs][:, electrode_idx, study_smin:study_smax]  # shape (n_epochs, n_times)
#         study_pipe = study_pipes[study_fold]
#         preds = study_pipe.predict_proba(study_data)[:, 1]
#         np.testing.assert_allclose(preds, study_previous_outcomes.decoder_proba.values)

#     for site in rows.itertuples():
#         for study_fold in range(len(study_pipes)):
#             study_previous_outcomes = study_dec_outcomes[study_dec_outcomes.fold == study_fold]
#             study_idxs = study_previous_outcomes.epoch_idx

#             study_pipe = study_pipes[study_fold]

#             # behavior decoding window and phonetic decoding window may be mismatched. match to the right window and move backward
#             transfer_smax = site.behavior_smax
#             transfer_smin = site.behavior_smax - study_pipe.n_features_in_
#             if transfer_smin < 0:
#                 raise ValueError(f"Cannot transfer to behavior window {site.behavior_smin}-{site.behavior_smax} with decoder input size {study_pipe.n_features_in_}")

#             study_data = data_i[study_idxs][:, site.electrode_idx, :]  # shape (n_epochs, n_times)

#             rolling_data = np.lib.stride_tricks.sliding_window_view(
#                 study_data,
#                 window_shape=study_pipe.n_features_in_,
#                 axis=-1
#             )
#             rolling_data = rolling_data[:, ::rolling_stride, :].transpose(1, 0, 2)  # shape (n_windows, n_epochs, window_size)
#             # compute window smin and smax for each rolling window
#             rolling_smax = np.arange(study_pipe.n_features_in_, data_i.shape[2] + 1, rolling_stride)
#             rolling_smin = rolling_smax - study_pipe.n_features_in_

#             # DEV manually normalize
#             rolling_data = rolling_data.copy()
#             rolling_data -= rolling_data.mean(axis=1, keepdims=True)
#             rolling_data /= rolling_data.std(axis=1, keepdims=True) + 1e-6

#             for window_j, smin_j, smax_j in zip(rolling_data, rolling_smin, rolling_smax):
#                 preds = study_pipe.steps[1][1].predict_proba(window_j)[:, 1]

#                 df = pd.DataFrame({
#                     "subject": site.subject,
#                     "electrode_idx": site.electrode_idx,
#                     "phoneme_pair": site.phoneme_pair,
#                     "smin": smin_j,
#                     "smax": smax_j,
#                     "behavior_smin": site.behavior_smin,
#                     "behavior_smax": site.behavior_smax,
#                     "fold": study_fold,
#                     "decoder_proba": preds,
#                     "decoder_target": study_previous_outcomes.decoder_target.values,
#                     "orig_decoder_proba": study_previous_outcomes.decoder_proba.values,
#                     "epoch_idx": study_idxs,
#                     "word_end": site.word_end,
#                 })

#                 rolling_transfer_results.append(df)

In [ ]:
# rolling_transfer_results_df = pd.concat(rolling_transfer_results, ignore_index=True)

In [ ]:
# rolling_roc_aucs = []
# for (subject, electrode_idx, phoneme_pair, word_end, smin, smax, behavior_smin, behavior_smax), group in tqdm(rolling_transfer_results_df.groupby(["subject", "electrode_idx", "phoneme_pair", "word_end", "smin", "smax", "behavior_smin", "behavior_smax"])):
#     for fold, group_i in group.groupby("fold"):
#         from sklearn.metrics import roc_auc_score
#         rolling_roc_aucs.append({
#             "subject": subject,
#             "electrode_idx": electrode_idx,
#             "phoneme_pair": phoneme_pair,
#             "word_end": word_end,
#             "smin": smin,
#             "smax": smax,
#             "behavior_smin": behavior_smin,
#             "behavior_smax": behavior_smax,
#             "fold": fold,
#             "rolling_roc_auc": roc_auc_score(group_i.decoder_target, group_i.decoder_proba),
#             "original_roc_auc": roc_auc_score(group_i.decoder_target, group_i.orig_decoder_proba),
#         })

In [ ]:
# rolling_roc_auc_df = pd.DataFrame(rolling_roc_aucs) \
#     .groupby(["subject", "electrode_idx", "phoneme_pair", "word_end", "smin", "smax", "behavior_smin", "behavior_smax"])[["rolling_roc_auc", "original_roc_auc"]].mean().sort_values("rolling_roc_auc", ascending=False)
# rolling_roc_auc_df["diff"] = rolling_roc_auc_df["rolling_roc_auc"] - rolling_roc_auc_df["original_roc_auc"]
# rolling_roc_auc_df = rolling_roc_auc_df.reset_index()
# rolling_roc_auc_df["smax_relative_to_phonetic_window"] = rolling_roc_auc_df.smax - 70  # phonetic window ends at 70
# rolling_roc_auc_df["smax_relative_to_behavior_window"] = rolling_roc_auc_df.smax - rolling_roc_auc_df.behavior_smax

In [ ]:
# sns.lineplot(data=rolling_roc_auc_df, x="smax_relative_to_phonetic_window", hue="phoneme_pair", y="rolling_roc_auc")